# Lambda Functions

## Introduction

Lambda functions are anonymous, single-expression functions written inline. They're most useful when you need to apply a quick transformation to every element in a Series — a common pattern in data cleaning.

## Objectives

You will be able to:

- Write lambda functions and explain when to use them over a named `def`
- Apply lambdas to a pandas Series using `.map()`
- Use lambdas with conditionals and as sort keys
- Use the `%` and `//` operators for row/column index arithmetic

In [ ]:
import pandas as pd

df = pd.read_csv('data/lambda_functions/Yelp_Reviews.csv', index_col=0)
df.head(2)

---

## Lambda Functions

The syntax is:

```python
lambda <argument>: <expression>
```

The expression is evaluated and returned automatically — no `return` keyword needed. Use `.map()` to apply a lambda to every element of a Series:

```python
series.map(lambda x: <expression>)
```

The argument name (`x`, `val`, `review_text`) is arbitrary — use whatever is most readable.

In [ ]:
# Count words in each review — both lines do exactly the same thing
df['text'].map(lambda x: len(x.split())).head()

# The argument name doesn't matter:
# df['text'].map(lambda review_text: len(review_text.split())).head()

---

## Practice: Transforming Columns

In [ ]:
# Square the stars column
df['stars'].map(lambda x: x ** 2).head()

In [ ]:
# Extract the month from a date string like '2012-11-13'
df['date'].map(lambda x: x[5:7]).head()

In [ ]:
# Average number of words across all reviews — one line
df['text'].map(lambda x: len(x.split())).mean()

In [ ]:
# Add a word count column, then inspect
df['word_count'] = df['text'].map(lambda x: len(x.split()))
df[['stars', 'text', 'word_count']].head()

---

## Lambda Functions with Conditionals

A ternary conditional fits inside a lambda:

```python
lambda x: 'A' if x > 90 else 'B'
```

Chained ternaries work but get hard to read quickly — if the logic needs more than two branches, a named `def` is cleaner.

In [ ]:
# Classify reviews as positive or negative based on keyword presence
positive_words = ['awesome', 'love', 'good', 'great']
df['text'].map(
    lambda x: 'Positive' if any(w in x.lower() for w in positive_words) else 'Negative'
).value_counts()

### Practice: Review length classifier

Rewrite the function below as a lambda and apply it to the `word_count` column to create a new `review_length` column.

In [ ]:
# Named function — to be rewritten as a lambda:
def categorise_length(n):
    if n < 50:
        return 'Short'
    elif n < 80:
        return 'Medium'
    else:
        return 'Long'

df['review_length'] = df['word_count'].map(
    lambda x: 'Short' if x < 50 else ('Medium' if x < 80 else 'Long')
)
df['review_length'].value_counts(normalize=True).round(2)

---

## Lambda as a Sort Key

Python's built-in `sorted()` accepts a `key=` argument — a function applied to each element before comparison. A lambda works well here.

In [ ]:
names = ['Miriam Marks', 'Sidney Baird', 'Elaine Barrera', 'Eddie Reeves',
         'Marley Beard', 'Jaiden Liu', 'Bethany Martin', 'Stephen Rios',
         'Audrey Mayer', 'Kameron Davidson', 'Carter Wong', 'Teagan Bennett']

# Default: alphabetical by first name
print('By first name:', sorted(names)[:3], '...')

# With key: alphabetical by last name
print('By last name: ', sorted(names, key=lambda x: x.split()[1])[:3], '...')

---

## The `%` and `//` Operators

These two operators are handy when iterating over a flat list and need to map each item to a row and column position.

| Operator | Name | What it does |
|----------|------|--------------|
| `%` | Modulus | Returns the **remainder** after division — useful for "every Nth element" |
| `//` | Floor division | Returns the **quotient** rounded down — useful for group membership |

In [ ]:
# % cycles through 0..N-1 repeatedly — useful for column index
for i in range(8):
    print(f'i={i}  i%4={i%4}')

In [ ]:
# // increments every N steps — useful for row index
for i in range(8):
    print(f'i={i}  i//4={i//4}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Arrange 12 subplots in a 3×4 grid using // and %
fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(12, 9))
x = np.linspace(-5, 5, 200)

for i in range(12):
    row, col = i // 4, i % 4
    axes[row, col].plot(x, x ** i)
    axes[row, col].set_title(f'x^{i}')

plt.tight_layout()
plt.show()

---

## Approach: Solve for One Case First, Then Generalise

Before writing a lambda to transform an entire column, prototype on a single value. This keeps the logic clear and makes edge cases obvious.

In [ ]:
# Step 1 — pick a single example
example = df['text'].iloc[0]
print(example[:80], '...')

# Step 2 — solve for that one case
words = example.split()
print(f'\nWord count: {len(words)}')

In [ ]:
# Step 3 — check an edge case: does extra spacing break things?
# split() handles multiple spaces correctly:
len('this  has   extra   spaces'.split())  # still counts 4 words

In [ ]:
# Step 4 — generalise to the full column
df['text'].map(lambda x: len(x.split())).head()

### Practice: Date reformat

Reorder each date in the `date` column from `YYYY-MM-DD` to `DD-MM-YYYY` using a lambda. Prototype on a single value first.

In [ ]:
# Prototype on one value
sample = df['date'].iloc[0]
print('Before:', sample)
print('After: ', f"{sample[8:10]}-{sample[5:7]}-{sample[:4]}")

In [ ]:
# Apply to the full column
df['date'].map(lambda x: f"{x[8:10]}-{x[5:7]}-{x[:4]}").head()

---

## Summary

In this notebook you learned how to:

- Write lambda functions and apply them to pandas Series with `.map()`
- Embed conditionals inside lambdas using ternary syntax
- Use lambdas as sort keys with `sorted()`
- Use `%` and `//` to compute row/column positions for subplot grids
- Prototype on a single value before generalising to a full column

Next: [03 — Missing Data](03_missing_data.ipynb)